<a href="https://colab.research.google.com/github/darksister1331/ActionBoard-AI/blob/main/Sentiment%2Btopic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **NEW**

Load all pips

In [1]:
!pip install cohere nltk scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.7 MB/s eta 0:00:00


Import

In [2]:
import re
import numpy as np
import pandas as pd
import cohere
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Load and Preprocess Transcript

In [3]:
# Load transcript file
with open('/content/transcript.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Extract transcript chunks using timestamp pattern
chunks = re.findall(r"\[\d+\.\d+-\d+\.\d+\](.*?)\n", raw_text, re.DOTALL)
chunks = [chunk.strip().lower() for chunk in chunks if chunk.strip()]

# Clean text function
def clean_text(text):
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return " ".join([w for w in text.split() if w not in stop_words])

cleaned_chunks = [clean_text(chunk) for chunk in chunks]


Perform LDA Modelling

In [4]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(cleaned_chunks)

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda_topics = lda.fit_transform(X)

topic_df = pd.DataFrame(lda_topics, columns=[f"Topic_{i}" for i in range(5)])


Bhattacharyya Distance Function

In [5]:
def bhattacharyya(p, q):
    p = np.array(p) / np.sum(p)
    q = np.array(q) / np.sum(q)
    p = np.where(p == 0, 1e-10, p)
    q = np.where(q == 0, 1e-10, q)
    return -np.log(np.sum(np.sqrt(p * q)))


Similarity between Query and Transcript Chunks

In [6]:
def compute_distance(user_query, vectorizer, lda_model):
    cleaned = clean_text(user_query)
    vec = vectorizer.transform([cleaned])
    user_topic = lda_model.transform(vec)[0]

    distances = []
    for idx, row in topic_df.iterrows():
        dist = bhattacharyya(user_topic, row.values)
        distances.append((idx, dist))

    return sorted(distances, key=lambda x: x[1])  # lower = more similar


Cohere API

In [7]:
co = cohere.Client("tl4K8mFoScI9bZpaFra9EnIXdNlWGKm3LDz1748p")  # 🔐 Replace with your actual API key


Search and Summarize

In [9]:
def search_and_summarize_by_topic(threshold=0.6):
    query = input(" Enter a topic to search for (e.g., 'organization structure'): ").strip().lower()
    distances = compute_distance(query, vectorizer, lda)
    relevant = [(i, d) for i, d in distances if d < threshold]

    if not relevant:
        print("\n No relevant chunks found for this topic.")
        return

    relevant_texts = [chunks[i] for i, _ in relevant]
    combined_text = "\n".join(relevant_texts)

    print(f"\n Found {len(relevant_texts)} relevant chunks. Summarizing now...\n")

    response = co.summarize(
        text=combined_text,
        length='medium',
        format='paragraph',
        model='summarize-xlarge'  #  supported summarization model
    )

    print(" Summary:\n")
    print(response.summary)


Example

In [25]:
search_and_summarize_by_topic()


 Enter a topic to search for (e.g., 'organization structure'): organisation

 Found 733 relevant chunks. Summarizing now...

 Summary:

The group continues working on the project, discussing different aspects and potential improvements. They go over the current status of the application and the progress made on the login and sign-up functionality. They also address some technical issues and discuss the need for further testing and refinement. They plan to meet again soon to review the updated code and ensure that all the components are working correctly. Additionally, they consider potential future enhancements to the application.


### **NEW_2**

In [11]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [12]:
sentiments = []

for chunk in chunks:
    score = sia.polarity_scores(chunk)
    sentiments.append({
        "text": chunk,
        "compound": score['compound'],  # Overall sentiment
        "pos": score['pos'],
        "neu": score['neu'],
        "neg": score['neg']
    })

sentiment_df = pd.DataFrame(sentiments)


In [13]:
def search_by_sentiment(sentiment='positive', threshold=0.4):
    if sentiment == 'positive':
        results = sentiment_df[sentiment_df['compound'] > threshold]
    elif sentiment == 'negative':
        results = sentiment_df[sentiment_df['compound'] < -threshold]
    else:  # neutral
        results = sentiment_df[(sentiment_df['compound'] <= threshold) & (sentiment_df['compound'] >= -threshold)]

    if results.empty:
        print(f"No {sentiment} chunks found.")
    else:
        print(f"\n {len(results)} {sentiment} chunks found:\n")
        for i, row in results.iterrows():
            print(f"[Sentiment Score: {row['compound']:.2f}]")
            print(row['text'])
            print("---\n")


In [14]:
search_by_sentiment('negative')   # or 'negative' or 'neutral'



 9 negative chunks found:

[Sentiment Score: -0.54]
you can use my bad days.
---

[Sentiment Score: -0.40]
when i think with the date of birth problem, something, let me check 99.
---

[Sentiment Score: -0.40]
i think we have some error in the.
---

[Sentiment Score: -0.50]
i don't have to share anything like in the screen, but i need to... what i was doing... i already have integrated zoom, but the problem is to fetch previous meetings, i cannot do it with my account.
---

[Sentiment Score: -0.40]
so you need to, the meetings need to be a part of that organization to be here to be disliked here.
---

[Sentiment Score: -0.56]
so it's pretty like one kind of like, you know, neck, like nasty, nasty, we call it a nasty loop, you know, nasty thing, you know.
---

[Sentiment Score: -0.40]
and if you get any bugs or any error, you just write down and let me know or let the juror know.
---

[Sentiment Score: -0.40]
the problem is if we do not host, if we use whisper locally, then
---

[Senti

In [15]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [16]:
sentiments = []

for chunk in chunks:
    score = sia.polarity_scores(chunk)
    sentiments.append({
        "text": chunk,
        "compound": score['compound'],  # Overall sentiment score
        "pos": score['pos'],
        "neu": score['neu'],
        "neg": score['neg']
    })

sentiment_df = pd.DataFrame(sentiments)


In [17]:
def search_by_sentiment_input(threshold=0.3):
    sentiment = input(" Enter sentiment to search for (positive / negative / neutral): ").strip().lower()

    if sentiment == 'positive':
        results = sentiment_df[sentiment_df['compound'] > threshold]
    elif sentiment == 'negative':
        results = sentiment_df[sentiment_df['compound'] < -threshold]
    elif sentiment == 'neutral':
        results = sentiment_df[
            (sentiment_df['compound'] >= -threshold) &
            (sentiment_df['compound'] <= threshold)
        ]
    else:
        print(" Invalid sentiment. Please enter 'positive', 'negative', or 'neutral'.")
        return

    if results.empty:
        print(f"No {sentiment} chunks found.")
    else:
        print(f"\n {len(results)} {sentiment} chunks found:\n")
        for i, row in results.iterrows():
            print(f"[Sentiment Score: {row['compound']:.2f}]")
            print(row['text'])
            print("---\n")


In [27]:
search_by_sentiment_input()


 Enter sentiment to search for (positive / negative / neutral): positive

 187 positive chunks found:

1. [Sentiment Score: 0.44]
i'm good.
---

2. [Sentiment Score: 0.48]
yeah, okay.
---

3. [Sentiment Score: 0.84]
so basically, yeah, so me and the like your rule, we had a like meeting yesterday and we just, you know, worked together and just able to api and today i also worked and i just finished like
---

4. [Sentiment Score: 0.36]
signing in and like sign up and you know, or maybe very few questions.
---

5. [Sentiment Score: 0.53]
stavocal, let me share share you.
---

6. [Sentiment Score: 0.35]
so let me share my skin.
---

7. [Sentiment Score: 0.48]
so as you, can you allow me to share my screen?
---

8. [Sentiment Score: 0.36]
thank you.
---

9. [Sentiment Score: 0.51]
so yeah, i just, you know, this is our, when somebody hit in our url, so he will be reaching this our home of our website.
---

10. [Sentiment Score: 0.72]
and here is like advanced meeting until against staff an

,text,compound,pos,neu,neg
0,i'm good.,0.4404,0.744,0.256,0.0
1,"yeah, okay.",0.4767,1.000,0.000,0.0
2,"so basically, yeah, so me and the like your ru...",0.8393,0.244,0.756,0.0
3,"signing in and like sign up and you know, or m...",0.3612,0.161,0.839,0.0
4,"stavocal, let me share share you.",0.5267,0.524,0.476,0.0
...,...,...,...,...,...
182,very good.,0.4927,0.762,0.238,0.0
183,we really appreciate.,0.4576,0.599,0.401,0.0
184,very good.,0.4927,0.762,0.238,0.0
185,thank you.,0.3612,0.714,0.286,0.0


### **NEW_3**

In [19]:
# Get only negative chunks based on threshold
negative_chunks = sentiment_df[sentiment_df['compound'] < -0.3]['text'].tolist()
combined_text = "\n".join(negative_chunks)


In [20]:
import cohere

co = cohere.Client("tl4K8mFoScI9bZpaFra9EnIXdNlWGKm3LDz1748p")  # Replace with your actual key

response = co.summarize(
    text=combined_text,
    length='medium',
    format='paragraph',
    model='summarize-xlarge'
)

print("🧠 Summary of negative sentiment:\n")
print(response.summary)


🧠 Summary of negative sentiment:

The speaker is facing an issue where they are unable to fetch previous meetings from their account and share their screen with a user. They are looking for assistance in troubleshooting the issue. They also want to know if there will be any issues with the 30 minute demonstration on Sunday and if there is a possibility to host using whisper locally to make it easier to integrate it into the backend.


In [21]:
def search_by_sentiment_input(threshold=0.3):
    sentiment = input(" Enter sentiment to search for (positive / negative / neutral): ").strip().lower()

    if sentiment == 'positive':
        results = sentiment_df[sentiment_df['compound'] > threshold]
    elif sentiment == 'negative':
        results = sentiment_df[sentiment_df['compound'] < -threshold]
    elif sentiment == 'neutral':
        results = sentiment_df[
            (sentiment_df['compound'] >= -threshold) &
            (sentiment_df['compound'] <= threshold)
        ]
    else:
        print("Invalid sentiment. Please enter 'positive', 'negative', or 'neutral'.")
        return pd.DataFrame()

    if results.empty:
        print(f"No {sentiment} chunks found.")
    else:
        print(f"\n {len(results)} {sentiment} chunks found:\n")
        for idx, (i, row) in enumerate(results.iterrows(), 1):
            print(f"{idx}. [Sentiment Score: {row['compound']:.2f}]")
            print(row['text'])
            print("---\n")

    return results.reset_index(drop=True)  # reset index for easy selection


In [22]:
def summarize_selected_chunk(results_df, sentiment_label='negative'):
    if results_df.empty:
        print("No data to summarize.")
        return

    try:
        idx = int(input(f" Enter the index of the chunk (1 to {len(results_df)}): ")) - 1
        selected_chunk = results_df.loc[idx, 'text']
    except:
        print(" Invalid selection.")
        return

    prompt = f"""This transcript chunk was labeled as having a {sentiment_label} sentiment.
Explain briefly why this might be the case based on the content:\n\n{selected_chunk}"""

    response = co.summarize(
        text=prompt,
        length='short',
        format='paragraph',
        model='summarize-xlarge'
    )

    print("\n Explanation for sentiment:\n")
    print(response.summary)


Search the sentiment chunk and reasoning for that sentiment

In [28]:
# Step 1: Run and capture the results
results = search_by_sentiment_input()

# Step 2: Select one chunk to summarize
summarize_selected_chunk(results, sentiment_label='negative')  # change label if needed



 Enter sentiment to search for (positive / negative / neutral): negative

 12 negative chunks found:

1. [Sentiment Score: -0.54]
you can use my bad days.
---

2. [Sentiment Score: -0.40]
when i think with the date of birth problem, something, let me check 99.
---

3. [Sentiment Score: -0.40]
i think we have some error in the.
---

4. [Sentiment Score: -0.50]
i don't have to share anything like in the screen, but i need to... what i was doing... i already have integrated zoom, but the problem is to fetch previous meetings, i cannot do it with my account.
---

5. [Sentiment Score: -0.40]
so you need to, the meetings need to be a part of that organization to be here to be disliked here.
---

6. [Sentiment Score: -0.38]
that is the restricted scope we decided.
---

7. [Sentiment Score: -0.56]
so it's pretty like one kind of like, you know, neck, like nasty, nasty, we call it a nasty loop, you know, nasty thing, you know.
---

8. [Sentiment Score: -0.40]
and if you get any bugs or any erro